### Import Dependecies

In [39]:
import uuid
import httpx

from a2a.client import create_client, ClientConfig, A2ACardResolver, ClientFactory
from a2a.types import (
    AgentCard,
    Message,
    Part,
    Role,
    SendMessageRequest,
)
from a2a.helpers import new_text_message, get_stream_response_text

In [5]:
BASE_URL = "http://localhost:10001"
PUBLIC_AGENT_CARD_PATH = "/.well-known/agent-card.json"

In [11]:
async with httpx.AsyncClient() as httpx_client:
    # Initialize A2ACardResolver
    resolver = A2ACardResolver(
        httpx_client=httpx_client,
        base_url=BASE_URL,
    )

    try:
        print(f"Fetching public agent card from: {BASE_URL}{PUBLIC_AGENT_CARD_PATH}")
        _public_card = await resolver.get_agent_card()
        print(f"Fetched public agent card")
        print(_public_card)

        final_agent_card_to_use = _public_card

    except Exception as e:
        print(f"Error fetching public agent card: {e}")
        raise RuntimeError("Failed to fetch public agent card")

Fetching public agent card from: http://localhost:10001/.well-known/agent-card.json
Fetched public agent card
name: "warehouse_manager_agent"
description: "An agent that can check the availability of items in the warehouses and reserve them."
supported_interfaces {
  url: "http://localhost:10001/"
  protocol_binding: "JSONRPC"
}
version: "1.0.0"
capabilities {
  streaming: true
}
default_input_modes: "text/plain"
default_output_modes: "text/plain"
skills {
  id: "ABC"
  name: "Check Availability"
  description: "Check the availability of items in the warehouses"
  tags: "warehouse"
  tags: "availability"
  examples: "what is the availability of the item 123?"
}
skills {
  id: "DEF"
  name: "Reserve Items"
  description: "Reserve items in the warehouses"
  tags: "warehouse"
  tags: "reservation"
  examples: "reserve 10 items of the item 123 in the Berlin warehouse."
}



In [ ]:
timeout_config = httpx.Timeout(
    connect=10.0,   # Connection timeout
    read=100.0,     # Read timeout (important for long-running operations)
    write=10.0,     # Write timeout
    pool=10.0       # Pool timeout
)
config = ClientConfig(streaming=False)

async with httpx.AsyncClient(timeout=timeout_config) as httpx_client:
    client = await create_client(agent=BASE_URL, client_config=config)
    print("A2AClient initialized")

    message = new_text_message(
        text="Hello, how are you?",
        role=Role.ROLE_USER,
    )
    request = SendMessageRequest(message=message)

    async for response in client.send_message(request):
        if response.HasField("message"):
            print(response.message)
        elif response.HasField("status_update"):
            print(response.status_update)
        elif response.HasField("artifact_update"):
            print(response.artifact_update)
        elif response.HasField("task"):
            print(response.task)

A2AClient initialized
id: "6537f647-96d8-4210-a5af-b065ebb4fc9d"
context_id: "1f594bb6-cf3f-473c-8650-9d20ff9e296a"
status {
  state: TASK_STATE_COMPLETED
  timestamp {
    seconds: 1779294634
    nanos: 582474000
  }
}
artifacts {
  artifact_id: "68e342a4-fd00-4543-b43b-444f9c3b4641"
  parts {
    text: "Hello! I’m doing well, thanks for asking. How can I help you today?"
  }
}
history {
  message_id: "004da102-c6d4-47bd-a7e8-84e941e51913"
  context_id: "1f594bb6-cf3f-473c-8650-9d20ff9e296a"
  task_id: "6537f647-96d8-4210-a5af-b065ebb4fc9d"
  role: ROLE_USER
  parts {
    text: "Hello, how are you?"
  }
}



In [21]:
response

task {
  id: "6537f647-96d8-4210-a5af-b065ebb4fc9d"
  context_id: "1f594bb6-cf3f-473c-8650-9d20ff9e296a"
  status {
    state: TASK_STATE_COMPLETED
    timestamp {
      seconds: 1779294634
      nanos: 582474000
    }
  }
  artifacts {
    artifact_id: "68e342a4-fd00-4543-b43b-444f9c3b4641"
    parts {
      text: "Hello! I’m doing well, thanks for asking. How can I help you today?"
    }
  }
  history {
    message_id: "004da102-c6d4-47bd-a7e8-84e941e51913"
    context_id: "1f594bb6-cf3f-473c-8650-9d20ff9e296a"
    task_id: "6537f647-96d8-4210-a5af-b065ebb4fc9d"
    role: ROLE_USER
    parts {
      text: "Hello, how are you?"
    }
  }
}

In [ ]:
timeout_config = httpx.Timeout(
    connect=10.0,   # Connection timeout
    read=100.0,     # Read timeout (important for long-running operations)
    write=10.0,     # Write timeout
    pool=10.0       # Pool timeout
)
config = ClientConfig(streaming=False)

async with httpx.AsyncClient(timeout=timeout_config) as httpx_client:
    client = await create_client(agent=BASE_URL, client_config=config)
    print("A2AClient initialized")

    message = new_text_message(
        text="Hello, how are you?",
        role=Role.ROLE_USER,
    )
    print("Sending message")
    request = SendMessageRequest(message=message)
    final_response = None

    print('Response:')
    async for response in client.send_message(request):
        text = get_stream_response_text(response)
        if text:
            print(text)
            
            

A2AClient initialized
Sending message
Response:
Hello! I’m doing well, thanks for asking. How can I help you today?


In [ ]:
timeout_config = httpx.Timeout(
    connect=10.0,
    read=300.0,   # LLM agents can be slow — give it 5 minutes
    write=10.0,
    pool=10.0,
)

httpx_client = httpx.AsyncClient(timeout=timeout_config)

config = ClientConfig(httpx_client=httpx_client)
factory = ClientFactory(config=config)

resolver = A2ACardResolver(httpx_client=httpx_client, base_url=BASE_URL)
agent_card = await resolver.get_agent_card()

client = factory.create(card=agent_card)
print("A2AClient initialized")

message = new_text_message(
    text="What is the availability of B0BP68QCZ2 in all of the warehouses?",
    role=Role.ROLE_USER,
)

request = SendMessageRequest(message=message)
print("Response:")
async for response in client.send_message(request):
    text = get_stream_response_text(response)
    if text:
        print(text)

A2AClient initialized
Response:
B0BP68QCZ2 is available in all warehouses, and each warehouse can fulfill at least 1 unit.

Availability by warehouse:
- DE-BER-01 — Berlin Distribution Center: 15
- FR-LY0-01 — Lyon Regional Warehouse: 86
- DE-MUN-01 — Munich Logistics Hub: 42
- FR-PAR-01 — Paris Central Depot: 25
- FR-MAR-01 — Marseille Mediterranean Hub: 4
- DE-HAM-01 — Hamburg North Warehouse: 65

If you want, I can also help reserve a quantity from the best warehouse(s).
